# QA: Compare Polyfit Correction Strategies

Side-by-side comparison of two downsampled polyfit strategies.

**Columns:** Left 3 = Original (vx, vy, vz) | Right 3 = Crop-17.5 (vx, vy, vz)

| Row | Content | Colormap |
|-----|---------|----------|
| 1   | Piecewise correction (corr − uncorr, full-FOV, unmasked) | jet |
| 2   | Polyfit correction field (full-FOV, unmasked) | jet |
| 3   | Polyfit corrected velocity (uncorr + polyfit, tissue masked) | RdBu_r |
| 4   | Piecewise corrected velocity — GT reference (tissue masked) | RdBu_r |
| 5   | Uncorrected velocity (tissue masked) | RdBu_r |
| 6   | Magnitude / tissue mask | gray |

Rows 1, 4–5 are identical in both column groups (shared data);
only rows 2–3 differ between original and crop-17.5.

In [1]:
import platform
from pathlib import Path

import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
import pandas as pd
from scipy.ndimage import zoom

from vascular_superenhancement.utils.path_config import load_path_config, _PROJECT_ROOT

config_name = "local_mac" if platform.system() == "Darwin" else "all_patients"
pc = load_path_config(config_name)

WORKING_DIR = pc.working_dir
PATIENT_DATA_DIR = WORKING_DIR / "patient_data"
REPO_ROOT = _PROJECT_ROOT

# The two downsampled folders to compare
DS_FOLDER_ORIGINAL = "downsampled_full_fov_128x128x64"
DS_FOLDER_CROP     = "downsampled_full_fov_128x128x64_crop-17.5"

LABEL_ORIGINAL = "Original"
LABEL_CROP     = "Crop-17.5"

FRAME_INDEX = 4
N_AXIAL = 6

OUTPUT_DIR = Path("qa_pec_compare_polyfit_images")
OUTPUT_DIR.mkdir(exist_ok=True)

# --- Load splits and filter ---
splits_df = pd.read_csv(REPO_ROOT / "splits" / "splits_01-15-26.csv")

FILTER_SPLITS: list[str] | None = None          # e.g. ["test"]
FILTER_PATIENTS: list[str] | None = None         # e.g. ["Biswifo", "Balboloop"]

patients_df = splits_df[splits_df["split"].isin(["train", "validation", "test"])].copy()
if FILTER_SPLITS:
    patients_df = patients_df[patients_df["split"].isin(FILTER_SPLITS)]
if FILTER_PATIENTS:
    patients_df = patients_df[patients_df["patient_id"].isin(FILTER_PATIENTS)]
patients_df = patients_df.sort_values(["split", "patient_id"]).reset_index(drop=True)

print(f"Patient data dir: {PATIENT_DATA_DIR}")
print(f"Original folder:  {DS_FOLDER_ORIGINAL}")
print(f"Crop folder:      {DS_FOLDER_CROP}")
print(f"Output dir:       {OUTPUT_DIR.resolve()}")
print(f"Patients to process: {len(patients_df)}")
print(patients_df["split"].value_counts())

Found project root at: /Users/yakhilesh/Files/7_PhD/vascular-superenhancement/code-base/vascular-superenhancement-4d-flow
Patient data dir: /Users/yakhilesh/Files/7_PhD/vascular-superenhancement/code-base/vascular-superenhancement-4d-flow/working_dir/all_patients/patient_data
Original folder:  downsampled_full_fov_128x128x64
Crop folder:      downsampled_full_fov_128x128x64_crop-17.5
Output dir:       /Users/yakhilesh/Files/7_PhD/vascular-superenhancement/code-base/vascular-superenhancement-4d-flow/notebooks/data-qa/qa_pec_compare_polyfit_images
Patients to process: 215
split
train         165
test           28
validation     22
Name: count, dtype: int64


In [2]:
COMPONENTS = ["vx", "vy", "vz"]
PLANE_DIM = {"axial": 2, "coronal": 1}


def _take_slice(vol: np.ndarray, plane: str, slice_idx: int | None) -> np.ndarray:
    dim = PLANE_DIM[plane]
    if slice_idx is None:
        slice_idx = vol.shape[dim] // 2
    slicing = [slice(None)] * vol.ndim
    slicing[dim] = slice_idx
    s = vol[tuple(slicing)]
    if plane == "coronal":
        s = s[::-1, ::-1]
    return s


def load_slice(nifti_path: Path, plane: str = "axial",
               slice_idx: int | None = None) -> np.ndarray:
    vol = nib.load(str(nifti_path)).get_fdata(dtype=np.float32)
    return _take_slice(vol, plane, slice_idx)


def _try_load_slice(nifti_path: Path, plane: str = "axial",
                    slice_idx: int | None = None) -> np.ndarray | None:
    if not nifti_path.exists():
        return None
    return load_slice(nifti_path, plane, slice_idx)


def load_mask_slice(patient_dir: Path, pid: str, ds_shape: tuple,
                    plane: str = "axial",
                    slice_idx: int | None = None) -> np.ndarray:
    mask_path = patient_dir / "nifti" / "velocity_correction" / f"correction_air_mask_{pid}.nii.gz"
    mask_vol = nib.load(str(mask_path)).get_fdata(dtype=np.float32)
    if mask_vol.shape != ds_shape:
        scale = np.array(ds_shape) / np.array(mask_vol.shape)
        mask_vol = zoom(mask_vol, scale, order=0)
    return _take_slice(mask_vol, plane, slice_idx) > 0.5


def apply_mask(arr: np.ndarray, mask: np.ndarray) -> np.ma.MaskedArray:
    return np.ma.masked_where(~mask, arr)


def make_comparison_figure(
    pid: str, ds_root_orig: Path, ds_root_crop: Path, patient_dir: Path,
    frame: int, plane: str = "axial", slice_idx: int | None = None,
) -> plt.Figure:
    """Side-by-side comparison: left 3 cols = original, right 3 cols = crop-17.5."""

    mag_path = ds_root_orig / "4d_flow_mag" / f"4d_flow_mag_{pid}_frame_{frame:02d}.nii.gz"
    mag_slice = load_slice(mag_path, plane, slice_idx)
    ds_shape = nib.load(str(mag_path)).shape
    ref_shape_2d = mag_slice.shape

    # --- Shared data (identical in both folders) ---
    uncorr, corr_pw = {}, {}
    for comp in COMPONENTS:
        uncorr[comp] = _try_load_slice(
            ds_root_orig / f"4d_flow_{comp}" / f"4d_flow_{comp}_{pid}_frame_{frame:02d}.nii.gz",
            plane, slice_idx)
        corr_pw[comp] = _try_load_slice(
            ds_root_orig / f"4d_flow_{comp}_corr" / f"4d_flow_{comp}_corr_{pid}_frame_{frame:02d}.nii.gz",
            plane, slice_idx)
    has_uncorr = all(uncorr[c] is not None for c in COMPONENTS)
    has_corr_pw = all(corr_pw[c] is not None for c in COMPONENTS)

    # VENC for de-normalising polyfit corrections
    venc_path = patient_dir / "nifti" / "velocity_correction" / f"poly_coefficients_{pid}.npz"
    venc = float(np.load(str(venc_path))["venc"]) if venc_path.exists() else 1.0

    # --- Polyfit corrections from each folder ---
    def load_polyfit(ds_root):
        poly = {}
        for comp in COMPONENTS:
            p = ds_root / f"ground_truth_correction_{comp}_{pid}.nii.gz"
            if p.exists():
                s = load_slice(p, plane, slice_idx)
                if s.shape != ref_shape_2d:
                    s = zoom(s, np.array(ref_shape_2d) / np.array(s.shape), order=1)
                poly[comp] = s * venc
            else:
                poly[comp] = None
        return poly

    poly_orig = load_polyfit(ds_root_orig)
    poly_crop = load_polyfit(ds_root_crop)
    has_poly_orig = all(poly_orig[c] is not None for c in COMPONENTS)
    has_poly_crop = all(poly_crop[c] is not None for c in COMPONENTS)

    # Polyfit-corrected velocities
    vel_orig = {c: uncorr[c] + poly_orig[c] for c in COMPONENTS} if has_uncorr and has_poly_orig else {}
    vel_crop = {c: uncorr[c] + poly_crop[c] for c in COMPONENTS} if has_uncorr and has_poly_crop else {}
    has_vel_orig = bool(vel_orig)
    has_vel_crop = bool(vel_crop)

    # --- Tissue mask ---
    tissue = load_mask_slice(patient_dir, pid, ds_shape, plane, slice_idx)

    # Piecewise correction = corr - uncorr
    has_diff = has_uncorr and has_corr_pw
    diff = {c: corr_pw[c] - uncorr[c] for c in COMPONENTS} if has_diff else {}

    # --- Colour scales ---
    vel_max = 300.0
    corr_vals = []
    if has_diff:
        corr_vals.extend(diff[c].ravel() for c in COMPONENTS)
    if has_poly_orig:
        corr_vals.extend(poly_orig[c].ravel() for c in COMPONENTS)
    if has_poly_crop:
        corr_vals.extend(poly_crop[c].ravel() for c in COMPONENTS)
    corr_max = max(np.percentile(np.abs(np.concatenate(corr_vals)), 99), 1.0) if corr_vals else 1.0

    bg_color = "0.25"
    jet_cmap = plt.cm.jet.copy(); jet_cmap.set_bad(bg_color)
    vel_cmap = plt.cm.RdBu_r.copy(); vel_cmap.set_bad(bg_color)

    dim_char = "z" if plane == "axial" else "y"
    z_label = f"{plane} {dim_char}={slice_idx}" if slice_idx is not None else f"{plane} {dim_char}=mid"

    # 6 columns: [orig vx, orig vy, orig vz, crop vx, crop vy, crop vz]
    n_rows = 6
    n_cols = 6
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 4.5, n_rows * 4),
                             constrained_layout=True)
    fig.suptitle(f"{pid}  frame {frame:02d}  {z_label}", fontsize=16, fontweight="bold")

    # Column group headers
    for j, comp in enumerate(COMPONENTS):
        axes[0, j].set_title(f"{LABEL_ORIGINAL} {comp}", fontsize=11, fontweight="bold")
        axes[0, j + 3].set_title(f"{LABEL_CROP} {comp}", fontsize=11, fontweight="bold")

    def _plot_cell(ax, arr, cmap, vmin, vmax, masked=False):
        if arr is None:
            ax.text(0.5, 0.5, "N/A", transform=ax.transAxes,
                    ha="center", va="center", fontsize=12, color="white")
            return None
        disp = apply_mask(arr, tissue).T if masked else arr.T
        return ax.imshow(disp, origin="upper", cmap=cmap, vmin=vmin, vmax=vmax)

    # Row 0: Piecewise correction (corr − uncorr) — unmasked, jet (same both sides)
    row_label = "Piecewise Corr."
    for j, comp in enumerate(COMPONENTS):
        _plot_cell(axes[0, j], diff.get(comp), jet_cmap, -corr_max, corr_max)
        im = _plot_cell(axes[0, j + 3], diff.get(comp), jet_cmap, -corr_max, corr_max)
    axes[0, 0].set_ylabel(row_label, fontsize=12, fontweight="bold")
    if im is not None:
        fig.colorbar(im, ax=axes[0, :].tolist(), fraction=0.015, pad=0.02, label="correction")

    # Row 1: Polyfit correction field — unmasked, jet
    row_label = "Polyfit Corr."
    for j, comp in enumerate(COMPONENTS):
        _plot_cell(axes[1, j], poly_orig.get(comp), jet_cmap, -corr_max, corr_max)
        im = _plot_cell(axes[1, j + 3], poly_crop.get(comp), jet_cmap, -corr_max, corr_max)
    axes[1, 0].set_ylabel(row_label, fontsize=12, fontweight="bold")
    if im is not None:
        fig.colorbar(im, ax=axes[1, :].tolist(), fraction=0.015, pad=0.02, label="correction")

    # Row 2: Polyfit corrected velocity — masked, RdBu_r
    row_label = "Polyfit Corr. Vel."
    for j, comp in enumerate(COMPONENTS):
        _plot_cell(axes[2, j], vel_orig.get(comp), vel_cmap, -vel_max, vel_max, masked=True)
        im = _plot_cell(axes[2, j + 3], vel_crop.get(comp), vel_cmap, -vel_max, vel_max, masked=True)
    axes[2, 0].set_ylabel(row_label, fontsize=12, fontweight="bold")
    if im is not None:
        fig.colorbar(im, ax=axes[2, :].tolist(), fraction=0.015, pad=0.02, label="velocity")

    # Row 3: Piecewise corrected velocity (GT) — masked, RdBu_r (same both sides)
    row_label = "Piecewise Corr. Vel. (GT)"
    for j, comp in enumerate(COMPONENTS):
        _plot_cell(axes[3, j], corr_pw.get(comp), vel_cmap, -vel_max, vel_max, masked=True)
        im = _plot_cell(axes[3, j + 3], corr_pw.get(comp), vel_cmap, -vel_max, vel_max, masked=True)
    axes[3, 0].set_ylabel(row_label, fontsize=12, fontweight="bold")
    if im is not None:
        fig.colorbar(im, ax=axes[3, :].tolist(), fraction=0.015, pad=0.02, label="velocity")

    # Row 4: Uncorrected velocity — masked, RdBu_r (same both sides)
    row_label = "Uncorrected Vel."
    for j, comp in enumerate(COMPONENTS):
        _plot_cell(axes[4, j], uncorr.get(comp), vel_cmap, -vel_max, vel_max, masked=True)
        im = _plot_cell(axes[4, j + 3], uncorr.get(comp), vel_cmap, -vel_max, vel_max, masked=True)
    axes[4, 0].set_ylabel(row_label, fontsize=12, fontweight="bold")
    if im is not None:
        fig.colorbar(im, ax=axes[4, :].tolist(), fraction=0.015, pad=0.02, label="velocity")

    # Row 5: Magnitude | Mask (shared)
    row_label = "Anatomy"
    axes[5, 0].imshow(mag_slice.T, origin="upper", cmap="gray")
    axes[5, 0].set_title("Mag", fontsize=10)
    axes[5, 1].imshow(tissue.T.astype(float), origin="upper", cmap="gray")
    axes[5, 1].set_title("Mask", fontsize=10)
    axes[5, 2].imshow(apply_mask(mag_slice, tissue).T, origin="upper", cmap="RdBu_r")
    axes[5, 2].set_title("Masked Mag", fontsize=10)
    for k in range(3, 6):
        axes[5, k].set_visible(False)
    axes[5, 0].set_ylabel(row_label, fontsize=12, fontweight="bold")

    for ax in axes.ravel():
        if ax.get_visible():
            ax.set_xticks([]); ax.set_yticks([]); ax.set_facecolor(bg_color)

    return fig

In [3]:
errors = []
total_images = 0

for idx, row in patients_df.iterrows():
    pid = row["patient_id"]
    split = row["split"]
    patient_dir = PATIENT_DATA_DIR / pid
    ds_root_orig = patient_dir / "nifti" / DS_FOLDER_ORIGINAL
    ds_root_crop = patient_dir / "nifti" / DS_FOLDER_CROP

    if not ds_root_orig.exists():
        print(f"[SKIP] {pid}: original downsampled dir not found")
        errors.append((pid, split, "missing original dir"))
        continue
    if not ds_root_crop.exists():
        print(f"[SKIP] {pid}: crop dir not found")
        errors.append((pid, split, "missing crop dir"))
        continue

    n_mag_frames = len(list((ds_root_orig / "4d_flow_mag").glob("*.nii.gz")))
    frame = min(FRAME_INDEX, n_mag_frames - 1)

    ref_vol = nib.load(str(ds_root_orig / "4d_flow_mag" / f"4d_flow_mag_{pid}_frame_{frame:02d}.nii.gz"))
    nx, ny, nz = ref_vol.shape

    axial_indices = np.linspace(0, nz - 1, N_AXIAL, dtype=int)
    coronal_idx = ny // 2

    slices = [(int(z), "axial") for z in axial_indices]
    slices.append((coronal_idx, "coronal"))

    out_dir = OUTPUT_DIR / pid
    out_dir.mkdir(exist_ok=True)

    patient_ok = True
    for si, plane in slices:
        dim_char = "z" if plane == "axial" else "y"
        out_path = out_dir / f"{plane}_{dim_char}{si:02d}.png"
        try:
            fig = make_comparison_figure(
                pid, ds_root_orig, ds_root_crop, patient_dir,
                frame=frame, plane=plane, slice_idx=si,
            )
            fig.savefig(out_path, dpi=120, bbox_inches="tight")
            plt.close(fig)
            total_images += 1
        except Exception as e:
            print(f"[ERR]  {pid} ({split}) {plane} {dim_char}={si}: {e}")
            errors.append((pid, split, f"{plane}_{dim_char}{si}: {e}"))
            patient_ok = False

    status = "[OK]  " if patient_ok else "[WARN]"
    print(f"{status} {pid} ({split}) — {len(slices)} images")

print(f"\nDone. {total_images} images across {len(patients_df)} patients.")
print(f"Output dir: {OUTPUT_DIR.resolve()}")
if errors:
    print(f"\n{len(errors)} errors/skips:")
    for pid, split, msg in errors:
        print(f"  {pid} ({split}): {msg}")

[OK]   Balboloop (test) — 7 images
[OK]   Biswifo (test) — 7 images
[OK]   Bomatog (test) — 7 images
[OK]   Boochuto (test) — 7 images
[OK]   Boumorim (test) — 7 images
[OK]   Bovutou (test) — 7 images
[OK]   Cadotueg (test) — 7 images
[OK]   Detodu (test) — 7 images
[OK]   Diecudey (test) — 7 images
[OK]   Diepami (test) — 7 images
[OK]   Diequipi (test) — 7 images
[OK]   Dithigog (test) — 7 images
[OK]   Dublafer (test) — 7 images
[OK]   Dujomal (test) — 7 images
[OK]   Elagieg (test) — 7 images
[OK]   Golotag (test) — 7 images
[OK]   Grequafie (test) — 7 images
[OK]   Gueshifa (test) — 7 images
[OK]   Kuquelok (test) — 7 images
[OK]   Oduskueb (test) — 7 images
[OK]   Quetode (test) — 7 images
[OK]   Runusath (test) — 7 images
[OK]   Sepigoo (test) — 7 images
[OK]   Stonscuetof (test) — 7 images
[OK]   Suquepog (test) — 7 images
[OK]   Tercippun (test) — 7 images
[OK]   Tiepolem (test) — 7 images
[OK]   Tisupey (test) — 7 images
[OK]   Amifer (train) — 7 images
[OK]   Aruborn (train